### Dataset Description

You are provided with a large number of Wikipedia comments which have been labeled by human raters for toxic behavior. The types of toxicity are:

- toxic
- severe_toxic
- obscene
- threat
- insult
- identity_hate

## Import Modules

In [6]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Load dataset
df = pd.read_csv("Reviews.csv")

df = df[['content','rating']]
df = df.dropna()

# ---------------- Sentiment ----------------
def sentiment_label(r):
    if r <= 2:
        return "Negative"
    elif r == 3:
        return "Neutral"
    else:
        return "Positive"

df["sentiment"] = df["rating"].apply(sentiment_label)

# ---------------- Factor Detection ----------------
def detect_factor(text):

    text = str(text).lower()

    if any(w in text for w in ["bill","billing","payment","invoice","refund","charge"]):
        return "Billing"

    elif any(w in text for w in ["staff","employee","manager","service","support","rude","helpful"]):
        return "Staff Behavior"

    elif any(w in text for w in ["clean","dirty","dust","hygiene","smell"]):
        return "Cleanliness"

    elif any(w in text for w in ["stock","available","availability","missing","out of stock"]):
        return "Product Availability"

    elif any(w in text for w in ["price","cost","expensive","cheap","value","worth"]):
        return "Pricing"

    elif any(w in text for w in ["wait","queue","line","delay","slow","late","waiting"]):
        return "Queue / Waiting Time"

    else:
        return "Others"

df["factor"] = df["content"].apply(detect_factor)

# Train model
X_train, X_test, y_train, y_test = train_test_split(
    df["content"], df["factor"], test_size=0.2, random_state=42
)

model = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english")),
    ("clf", LogisticRegression())
])

model.fit(X_train, y_train)

joblib.dump(model, "factor_model.joblib")

print("Model trained and saved successfully")

Model trained and saved successfully
